In [1]:
import pandas as pd
import yfinance as yf
from pathlib import Path

TICKER = "SYK"
RAW_DIR = Path("..")/"data"/"raw"

In [2]:
stock = yf.Ticker(TICKER)

In [3]:
income = stock.income_stmt
print(income.shape)
income

(46, 5)


,2025-12-31,2024-12-31,2023-12-31,2022-12-31,2021-12-31
Tax Effect Of Unusual Items,-4.775366e+07,-1.396114e+08,-4.968000e+06,-3.267000e+07,NaN
Tax Rate For Calcs,2.809040e-01,1.428980e-01,1.380000e-01,1.210000e-01,NaN
Normalized EBITDA,6.484000e+09,5.928000e+09,5.100000e+09,4.379000e+09,NaN
Total Unusual Items,-1.700000e+08,-9.770000e+08,-3.600000e+07,-2.700000e+08,NaN
Total Unusual Items Excluding Goodwill,-1.700000e+08,-9.770000e+08,-3.600000e+07,-2.700000e+08,NaN
Net Income From Continuing Operation Net Minority Interest,3.246000e+09,2.993000e+09,3.165000e+09,2.358000e+09,NaN
Reconciled Depreciation,1.193000e+09,1.050000e+09,1.028000e+09,9.980000e+08,NaN
Reconciled Cost Of Revenue,8.590000e+09,7.728000e+09,7.047000e+09,6.500000e+09,NaN
EBITDA,6.314000e+09,4.951000e+09,5.064000e+09,4.109000e+09,NaN
EBIT,5.121000e+09,3.901000e+09,4.036000e+09,3.111000e+09,NaN


In [9]:
today = pd.Timestamp.today().strftime("%Y-%m-%d")

# Create the two missing tables first (a variable must exist before you can use it)
balance = stock.balance_sheet
cashflow = stock.cashflow

income.to_csv(RAW_DIR / f"{TICKER}_income_downloaded_{today}.csv")
balance.to_csv(RAW_DIR / f"{TICKER}_balance_downloaded_{today}.csv")
cashflow.to_csv(RAW_DIR / f"{TICKER}_cashflow_downloaded_{today}.csv")

print("Saved income, balance sheet and cash flow for", TICKER)

Saved income, balance sheet and cash flow for SYK


In [7]:
revenue = income.loc["Total Revenue"]   # capital T: labels must match exactly
revenue

2025-12-31    2.511600e+10
2024-12-31    2.259500e+10
2023-12-31    2.049800e+10
2022-12-31    1.844900e+10
2021-12-31             NaN
Name: Total Revenue, dtype: float64

In [10]:
rev_now = revenue["2025-12-31"]
rev_start = revenue["2022-12-31"]

cagr = (rev_now / rev_start)**(1/3)-1

print(f"Stryker 3-year revenue CAGR : {cagr: .1%}")

Stryker 3-year revenue CAGR :  10.8%


In [ ]:
YEAR = "2025-12-31"

key_numbers = {
    "Revenue":             income.loc["Total Revenue", YEAR],
    "Operating income":    income.loc["Operating Income", YEAR],
    "Net income":          income.loc["Net Income", YEAR],
    "Operating cash flow": cashflow.loc["Operating Cash Flow", YEAR],
    "Capital expenditure": cashflow.loc["Capital Expenditure", YEAR],
    "Total debt":          balance.loc["Total Debt", YEAR],
    "Cash":                balance.loc["Cash And Cash Equivalents", YEAR],
    "Shares outstanding":  balance.loc["Ordinary Shares Number", YEAR],
}

key = pd.Series(key_numbers) / 1e6   # convert to millions, like the 10-K
key.round(0)

## Check against Stryker's 10-K (FY2025)

**Source:** Stryker Corp 10-K for the year ended 31 Dec 2025, filed with the SEC on 11 Feb 2026 (accession no. 0000310764-26-000010). Figures taken from the SEC's official XBRL data (`data.sec.gov`), in $ millions.

- **Total debt** in the 10-K is two balance-sheet lines: current debt (1,000) + long-term debt (14,859).
- **Shares outstanding** in the 10-K is from the cover page, dated 31 Jan 2026 (one month after the balance sheet), so a tiny difference is expected.
- **Rule:** a difference under 1% counts as a match.

In [ ]:
# Official 10-K figures, in $ millions (source: markdown cell above)
ten_k = pd.Series({
    "Revenue":             25116,
    "Operating income":    4889,
    "Net income":          3246,
    "Operating cash flow": 5044,
    "Capital expenditure": -761,         # 10-K shows 761 as a payment; negative = cash out
    "Total debt":          1000 + 14859,  # current debt + long-term debt
    "Cash":                4011,
    "Shares outstanding":  382.7,
})

# Put Yahoo and the 10-K side by side
check = pd.DataFrame({"Yahoo": key, "10-K": ten_k})
check["Difference %"] = (check["Yahoo"] / check["10-K"] - 1) * 100
check["Match?"] = check["Difference %"].abs() < 1   # True if within 1%

check.round(1)